In [0]:
import json
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import DataFrame, SparkSession, Row
from typing import List, Tuple

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


class DQCheckError(Exception):
    """Raised after all DQ rules are attempted, if one or more failed to apply."""
    pass


def load_dq_rules(spark: SparkSession, job_id: int) -> List[Row]:
    """
    Load DQ rules for a given job, ordered by Rule_ID.

    Args:
        spark: Active SparkSession used to run the query.
        job_id: The job these DQ rules apply to (matches Job_Id in DataQualityRule).

    Returns:
        List of Row objects, each exposing Rule_ID, Job_Id, Column_Name,
        Check_Type, and Rule_Parameters (JSON string or NULL).
    """
    return spark.sql(f"""
        SELECT * FROM ct_oil_gas.sc_metadata.DataQualityRule
        WHERE Job_Id = {job_id}
        ORDER BY Rule_ID
    """).collect()

class DQChecker:
    """
    Applies configured DQ rules to a DataFrame and splits it into
    valid and invalid DataFrames.
    """

    def __init__(self, df: DataFrame):
        """
        Args:
            df: The DataFrame to check. Mutated in place across method
                calls via self.df.
        """
        self.df = df
        self.flag_columns: List[str] = []
        self.rule_labels: dict = {}  # flag_col -> Check_Type

    def check_not_null(self, column_name: str) -> None:
        """Add a boolean flag column: True if column_name is not null."""
        flag_col = f"dq_{column_name}_not_null"
        self.df = self.df.withColumn(flag_col, F.col(column_name).isNotNull())
        self.flag_columns.append(flag_col)

    def check_in_range(self, column_name: str, min_value, max_value) -> None:
        """Add a boolean flag column: True if column_name is between min_value and max_value (inclusive)."""
        flag_col = f"dq_{column_name}_in_range"
        self.df = self.df.withColumn(flag_col, F.col(column_name).between(min_value, max_value))
        self.flag_columns.append(flag_col)

    def check_dedup(self, key_columns: List[str], order_by_column: str) -> None:
        """
        Add a boolean flag column: True if the row is the first occurrence
        per key_columns, ordered by order_by_column ascending.
        """
        flag_col = "dq_dedup_key"
        w = Window.partitionBy(*key_columns).orderBy(order_by_column)
        self.df = self.df.withColumn(flag_col, F.row_number().over(w) == 1)
        self.flag_columns.append(flag_col)

    def apply_config(self, config_rows: List[Row]) -> Tuple[DataFrame, DataFrame]:
        """
        Apply configured DQ rules and split the DataFrame into valid/invalid.
        """
        error_count = 0
        errors = []

        for row in config_rows:
            check_type = row["Check_Type"]
            col_name = row["Column_Name"]

            try:
                if check_type == "NOT_NULL":
                    self.check_not_null(col_name)
                elif check_type == "in_list":
                    params = json.loads(row["Rule_Parameters"])
                    self.check_in_range(col_name, params["min_value"], params["max_value"])
                elif check_type == "DEDUP_KEY":
                    params = json.loads(row["Rule_Parameters"])
                    key_columns = [c.strip() for c in col_name.split(",")]
                    self.check_dedup(key_columns, params["order_by"])
                else:
                    raise ValueError(f"Unknown Check_Type: {check_type}")

                self.rule_labels[self.flag_columns[-1]] = check_type

            except Exception as e:
                error_count += 1
                error_msg = f"Failed DQ rule [Rule_ID={row['Rule_ID']}, type={check_type}, column={col_name}]: {e}"
                errors.append(error_msg)
                logger.error(error_msg)

        if self.flag_columns:
            self.df = self.df.withColumn(
                "dq_overall_pass",
                F.array_min(F.array(*[F.col(c) for c in self.flag_columns]))
            )
            self.df = self.df.withColumn(
                "dq_failure_reasons",
                F.array_compact(F.array(*[
                    F.when(~F.col(c), F.lit(label)) for c, label in self.rule_labels.items()
                ]))
            )
        else:
            # No rules applied successfully — nothing to split on
            self.df = self.df.withColumn("dq_overall_pass", F.lit(True))
            self.df = self.df.withColumn("dq_failure_reasons", F.array().cast("array<string>"))

        if error_count > 0:
            summary = f"{error_count} DQ rule(s) failed to apply:\n" + "\n".join(errors)
            raise DQCheckError(summary)

        valid_df = self.df.filter(F.col("dq_overall_pass")).drop(
            *self.flag_columns, "dq_overall_pass", "dq_failure_reasons"
        )
        invalid_df = self.df.filter(~F.col("dq_overall_pass")).drop(*self.flag_columns, "dq_overall_pass")

        return valid_df, invalid_df

In [0]:
# catalog_name = "ct_oil_gas"
# bronze_schema = "sc_bronze"
# table_name = "oil_gas_transactions"

# config_rows = load_transformation_config(spark,1)

# df = spark.table(f"{catalog_name}.{bronze_schema}.oil_gas_transactions")

# transformer = Transformation(df)
# transformed_df = transformer.apply_config(config_rows)

In [0]:
# config_rows = load_dq_rules(spark, job_id=1)

# df = spark.table(f"{catalog_name}.{bronze_schema}.{table_name}")

# checker = DQChecker(df)
# valid_df, invalid_df = checker.apply_config(config_rows)  # raises DQCheckError if any rule failed to apply

# # store invalid rows separately (quarantine)
# invalid_df.write.format("delta").mode("append").saveAsTable(
#     f"{catalog_name}.{silver_schema}.{table_name}_dq_quarantine"
# )

# # proceed with valid rows
# valid_df.write.format("delta").mode("append").saveAsTable(
#     f"{catalog_name}.{silver_schema}.{table_name}_clean"
# )

# print(f"Valid rows: {valid_df.count()}, Invalid rows: {invalid_df.count()}")